In [3]:
#!/usr/bin/env python3
"""
Script de scraping des annonces immobilières de CoinAfrique Togo
VERSION AMÉLIORÉE: Extraction détaillée du quartier, ville et région
"""

import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urljoin
from datetime import datetime, timedelta

class CoinAfriqueScraperTogo:
    def __init__(self, output_file='coinafrique_60_pages.csv', debug=False):
        self.base_url = "https://tg.coinafrique.com"
        self.category_url = f"{self.base_url}/categorie/immobilier"
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9',
            'Accept-Language': 'fr-FR,fr;q=0.9,en-US;q=0.8,en;q=0.7',
        }
        self.session = requests.Session()
        self.session.headers.update(self.headers)
        self.all_listings = []
        self.output_file = output_file
        self.debug = debug
        
        # Listes des quartiers et villes connues du Togo
        self.quartiers_lome = [
            'Adidogomé', 'Agoè', 'Nyékonakpoè', 'Assiyéyé', 'Bè', 'Tokoin', 
            'Amoutivé', 'Cacavéli', 'Adakpamé', 'Légbassito', 'Hédzranawoé',
            'Aflao Gakli', 'Démakpoè', 'Kélégougan', 'Mission Tové', 'Akodésséwa',
            'Amahoutévé', 'Djidjolé', 'Agbalépédogan', 'Zanguéra', 'Kégué',
            'Hanoukopé', 'Sogbossito', 'Totsi', 'Sito Kpota', 'Adamavo',
            'Gbényédzi', 'Kodjoviakopé', 'Avédji', 'Avenou', 'Aképé'
        ]
        
        self.villes_togo = [
            'Lomé', 'Kara', 'Sokodé', 'Atakpamé', 'Kpalimé', 'Dapaong',
            'Tsévié', 'Aného', 'Bassar', 'Tabligbo', 'Vogan', 'Tchamba',
            'Badou', 'Notsé', 'Amlamé', 'Mango', 'Niamtougou', 'Sotouboua'
        ]
        
        self.regions_togo = [
            'Maritime', 'Plateaux', 'Centrale', 'Kara', 'Savanes'
        ]
        
    def normalize_date(self, date_text):
        """Convertit différents formats de date en format standard DD/MM/YYYY"""
        if not date_text or date_text == 'N/A':
            return 'N/A'
        
        date_text = date_text.strip().lower()
        today = datetime.now()
        
        try:
            # Pattern: "il y a X jours/heures/minutes"
            if 'il y a' in date_text or 'ya' in date_text:
                # Extraire le nombre
                num_match = re.search(r'(\d+)', date_text)
                if num_match:
                    num = int(num_match.group(1))
                    
                    if 'minute' in date_text or 'min' in date_text:
                        date_obj = today - timedelta(minutes=num)
                    elif 'heure' in date_text or 'hour' in date_text or 'h' in date_text:
                        date_obj = today - timedelta(hours=num)
                    elif 'jour' in date_text or 'day' in date_text or 'j' in date_text:
                        date_obj = today - timedelta(days=num)
                    elif 'semaine' in date_text or 'week' in date_text:
                        date_obj = today - timedelta(weeks=num)
                    elif 'mois' in date_text or 'month' in date_text:
                        date_obj = today - timedelta(days=num*30)
                    elif 'an' in date_text or 'year' in date_text:
                        date_obj = today - timedelta(days=num*365)
                    else:
                        return date_text  # Format non reconnu
                    
                    return date_obj.strftime('%d/%m/%Y')
            
            # Pattern: "aujourd'hui" ou "today"
            if "aujourd'hui" in date_text or 'today' in date_text:
                return today.strftime('%d/%m/%Y')
            
            # Pattern: "hier" ou "yesterday"
            if 'hier' in date_text or 'yesterday' in date_text:
                date_obj = today - timedelta(days=1)
                return date_obj.strftime('%d/%m/%Y')
            
            # Pattern: date au format DD/MM/YYYY ou DD-MM-YYYY
            date_match = re.search(r'(\d{1,2})[/-](\d{1,2})[/-](\d{2,4})', date_text)
            if date_match:
                day, month, year = date_match.groups()
                if len(year) == 2:
                    year = '20' + year
                return f"{day.zfill(2)}/{month.zfill(2)}/{year}"
            
            # Pattern: date au format YYYY-MM-DD
            date_match = re.search(r'(\d{4})[/-](\d{1,2})[/-](\d{1,2})', date_text)
            if date_match:
                year, month, day = date_match.groups()
                return f"{day.zfill(2)}/{month.zfill(2)}/{year}"
            
            # Pattern: mois en texte (ex: "15 janvier 2024", "Jan 15, 2024")
            mois_fr = {
                'janvier': '01', 'jan': '01',
                'février': '02', 'fév': '02', 'fevrier': '02', 'fev': '02',
                'mars': '03', 'mar': '03',
                'avril': '04', 'avr': '04',
                'mai': '05',
                'juin': '06',
                'juillet': '07', 'juil': '07',
                'août': '08', 'aout': '08',
                'septembre': '09', 'sep': '09', 'sept': '09',
                'octobre': '10', 'oct': '10',
                'novembre': '11', 'nov': '11',
                'décembre': '12', 'déc': '12', 'decembre': '12', 'dec': '12'
            }
            
            for mois_name, mois_num in mois_fr.items():
                if mois_name in date_text:
                    # Chercher le jour
                    day_match = re.search(r'(\d{1,2})', date_text)
                    # Chercher l'année
                    year_match = re.search(r'(\d{4})', date_text)
                    
                    if day_match:
                        day = day_match.group(1).zfill(2)
                        year = year_match.group(1) if year_match else str(today.year)
                        return f"{day}/{mois_num}/{year}"
            
            # Si aucun pattern ne correspond, retourner le texte original
            return date_text
            
        except Exception as e:
            if self.debug:
                print(f"[DEBUG] Erreur normalisation date '{date_text}': {e}")
            return date_text
    
    def extract_location_details(self, text):
        """Extrait quartier, ville et région depuis un texte"""
        if not text or text == 'N/A':
            return {'quartier': 'N/A', 'ville': 'N/A', 'region': 'N/A'}
        
        text_lower = text.lower()
        
        # Extraire le quartier
        quartier = 'N/A'
        for q in self.quartiers_lome:
            if q.lower() in text_lower:
                quartier = q
                break
        
        # Si pas de quartier trouvé dans la liste, chercher pattern "quartier XXX"
        if quartier == 'N/A':
            quartier_match = re.search(r'quartier\s+([A-ZÀ-Ÿa-zà-ÿ\s-]+?)(?:,|\.|$)', text, re.I)
            if quartier_match:
                quartier = quartier_match.group(1).strip()
        
        # Extraire la ville
        ville = 'N/A'
        for v in self.villes_togo:
            if v.lower() in text_lower:
                ville = v
                break
        
        # Extraire la région
        region = 'N/A'
        for r in self.regions_togo:
            if r.lower() in text_lower:
                region = r
                break
        
        # Si Lomé est détecté, région = Maritime par défaut
        if ville == 'Lomé' and region == 'N/A':
            region = 'Maritime'
        
        return {
            'quartier': quartier,
            'ville': ville,
            'region': region
        }
        
    def get_page_listings(self, url):
        """Récupère les annonces d'une page"""
        try:
            response = self.session.get(url, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            
            listings = []
            seen_urls = set()
            
            for link in soup.find_all('a', href=re.compile(r'/annonce/')):
                href = link.get('href')
                if href and '/annonce/' in href:
                    full_url = urljoin(self.base_url, href)
                    if full_url not in seen_urls:
                        seen_urls.add(full_url)
                        listings.append({'url': full_url})
            
            return listings, None
        except Exception as e:
            print(f"❌ Erreur: {e}")
            return [], None
    
    def extract_listing_details(self, url):
        """Extrait tous les détails d'une annonce"""
        try:
            time.sleep(1)
            response = self.session.get(url, timeout=15)
            response.raise_for_status()
            soup = BeautifulSoup(response.text, 'html.parser')
            
            details = {'url': url}
            
            title = soup.find('h1')
            details['titre'] = title.get_text(strip=True) if title else 'N/A'
            
            # Prix - Recherche améliorée
            prix = None
            
            if self.debug:
                print(f"\n[DEBUG] Recherche du prix pour: {url}")
            
            body_text = soup.get_text()
            
            patterns = [
                r'(\d[\d\s,.]*)\s*(?:CFA|FCFA|F\s*CFA)',
                r'(?:Prix\s*:?\s*)?(\d[\d\s,.]+)\s*(?:CFA|FCFA)',
                r'(\d+(?:[.,]\d+)?(?:\s*\d+)*)\s*(?:CFA|FCFA)',
            ]
            
            for pattern in patterns:
                price_match = re.search(pattern, body_text, re.I)
                if price_match:
                    prix_nombre = price_match.group(1).strip()
                    prix = f"{prix_nombre} CFA"
                    if self.debug:
                        print(f"[DEBUG] Pattern trouvé: {prix}")
                    break
            
            if not prix:
                price_elem = soup.find(['span', 'div', 'p', 'h2', 'h3', 'strong'], class_=re.compile(r'price|prix|amount|montant|cost', re.I))
                if price_elem:
                    text = price_elem.get_text(strip=True)
                    if re.search(r'CFA|FCFA', text, re.I):
                        prix = text
                        if self.debug:
                            print(f"[DEBUG] Méthode class: {prix}")
            
            if not prix:
                price_item = soup.find(['span', 'div', 'meta'], {'itemprop': 'price'})
                if price_item:
                    prix = price_item.get('content') or price_item.get_text(strip=True)
                    if not re.search(r'CFA', prix, re.I):
                        prix = f"{prix} CFA"
                    if self.debug:
                        print(f"[DEBUG] Méthode itemprop: {prix}")
            
            if not prix:
                price_patterns = soup.find_all(string=re.compile(r'\d+.*(?:CFA|FCFA)', re.I))
                for pattern in price_patterns:
                    text = pattern.strip()
                    if 5 < len(text) < 50 and re.search(r'\d', text):
                        match = re.search(r'(\d[\d\s,.]*)\s*(?:CFA|FCFA)', text, re.I)
                        if match:
                            prix_nombre = match.group(1).strip()
                            prix = f"{prix_nombre} CFA"
                            if self.debug:
                                print(f"[DEBUG] Méthode scan: {prix}")
                            break
            
            if not prix:
                price_meta = soup.find('meta', {'property': re.compile(r'price|amount', re.I)})
                if price_meta:
                    content = price_meta.get('content', '')
                    if content:
                        prix = f"{content} CFA" if not re.search(r'CFA', content, re.I) else content
                        if self.debug:
                            print(f"[DEBUG] Méthode meta: {prix}")
            
            if not prix:
                if self.debug:
                    print(f"[DEBUG] ❌ Aucun prix trouvé")
            
            details['prix'] = prix if prix else 'Prix sur demande'
            
            desc = soup.find(['div', 'p'], class_=re.compile(r'description|detail', re.I))
            if not desc:
                desc = soup.find('div', {'itemprop': 'description'})
            details['description'] = desc.get_text(strip=True) if desc else 'N/A'
            
            # ===== EXTRACTION AMÉLIORÉE DE LA LOCALISATION =====
            location_text = 'N/A'
            
            # Méthode 1: Chercher dans les balises avec class location/ville/quartier
            location = soup.find(['span', 'div', 'p', 'a'], class_=re.compile(r'location|ville|quartier|address|lieu', re.I))
            if location:
                location_text = location.get_text(strip=True)
            
            # Méthode 2: Chercher dans les balises avec itemprop
            if not location or location_text == 'N/A':
                location_item = soup.find(['span', 'div'], {'itemprop': re.compile(r'address|location', re.I)})
                if location_item:
                    location_text = location_item.get_text(strip=True)
            
            # Méthode 3: Chercher pattern "Localisation:" ou "Adresse:"
            if not location or location_text == 'N/A':
                loc_pattern = soup.find(string=re.compile(r'(?:Localisation|Adresse|Lieu)\s*:', re.I))
                if loc_pattern:
                    parent = loc_pattern.find_parent()
                    if parent:
                        location_text = parent.get_text(strip=True)
                        # Nettoyer le préfixe
                        location_text = re.sub(r'^(?:Localisation|Adresse|Lieu)\s*:\s*', '', location_text, flags=re.I)
            
            # Méthode 4: Chercher dans le titre
            if not location or location_text == 'N/A':
                # Chercher pattern "à XXX" ou "- XXX"
                loc_in_title = re.search(r'(?:à|À|-)\s*([A-ZÀ-Ÿ][a-zà-ÿ\s-]+?)(?:,|\.|$)', details['titre'])
                if loc_in_title:
                    location_text = loc_in_title.group(1).strip()
            
            # Méthode 5: Chercher dans la description
            if not location or location_text == 'N/A':
                desc_text = details.get('description', '')
                # Chercher pattern "situé à XXX" ou "localisé à XXX"
                loc_in_desc = re.search(r'(?:situé|localisé|se trouve|sis)\s+(?:à|au|dans)\s+([A-ZÀ-Ÿ][a-zà-ÿ\s-]+?)(?:,|\.|$)', desc_text, re.I)
                if loc_in_desc:
                    location_text = loc_in_desc.group(1).strip()
            
            # Méthode 6: Chercher directement les noms de villes/quartiers dans tout le texte
            if not location or location_text == 'N/A':
                full_text = soup.get_text()
                for ville in self.villes_togo:
                    if ville.lower() in full_text.lower():
                        location_text = ville
                        break
                
                if location_text == 'N/A':
                    for quartier in self.quartiers_lome:
                        if quartier.lower() in full_text.lower():
                            location_text = f"{quartier}, Lomé"
                            break
            
            details['localisation'] = location_text
            
            # ===== EXTRACTION DÉTAILLÉE: QUARTIER, VILLE, RÉGION =====
            location_details = self.extract_location_details(location_text + ' ' + details['titre'] + ' ' + details['description'])
            details['quartier'] = location_details['quartier']
            details['ville'] = location_details['ville']
            details['region'] = location_details['region']
            
            # Type d'offre
            offer_type = 'N/A'
            if 'vente' in details['titre'].lower():
                offer_type = 'Vente'
            elif 'location' in details['titre'].lower():
                offer_type = 'Location'
            details['type_offre'] = offer_type
            
            # Type de bien
            property_type = 'N/A'
            title_lower = details['titre'].lower()
            if 'villa' in title_lower:
                property_type = 'Villa'
            elif 'appartement' in title_lower:
                property_type = 'Appartement'
            elif 'terrain' in title_lower:
                property_type = 'Terrain'
            elif 'immeuble' in title_lower:
                property_type = 'Immeuble'
            elif 'bureau' in title_lower or 'commerce' in title_lower:
                property_type = 'Bureau/Commerce'
            elif 'chambre' in title_lower:
                property_type = 'Chambre'
            details['type_bien'] = property_type
            
            # Nombre de pièces
            pieces_match = re.search(r'(\d+)\s*(?:pièces?|chambres?)', details['titre'], re.I)
            details['nombre_pieces'] = pieces_match.group(1) if pieces_match else 'N/A'
            
            # Surface
            surface_match = re.search(r'(\d+(?:\.\d+)?)\s*(?:m²|m2|lots?)', details['titre'] + ' ' + details.get('description', ''), re.I)
            details['surface'] = surface_match.group(1) if surface_match else 'N/A'
            
            # Date de publication
            date_elem = soup.find(['span', 'div', 'time'], class_=re.compile(r'date|time|posted', re.I))
            date_brute = date_elem.get_text(strip=True) if date_elem else 'N/A'
            
            # Normaliser la date en format DD/MM/YYYY
            details['date_publication'] = self.normalize_date(date_brute)
            
            # ===== EXTRACTION AMÉLIORÉE DU NUMÉRO DE TÉLÉPHONE =====
            telephone = 'N/A'
            
            # Méthode 1: Chercher dans les liens tel:
            phone_link = soup.find(['a', 'button'], href=re.compile(r'tel:'))
            if phone_link:
                tel_href = phone_link.get('href', '')
                # Extraire le numéro depuis tel:+228XXXXXXXX
                tel_match = re.search(r'tel:\+?(\d+)', tel_href)
                if tel_match:
                    telephone = tel_match.group(1)
                else:
                    telephone = phone_link.get_text(strip=True)
            
            # Méthode 2: Chercher dans les balises avec class contenant 'phone', 'tel', 'contact', 'whatsapp'
            if telephone == 'N/A':
                phone_elem = soup.find(['span', 'div', 'p', 'a', 'button'], class_=re.compile(r'phone|tel|contact|whatsapp|numero', re.I))
                if phone_elem:
                    text = phone_elem.get_text(strip=True)
                    # Extraire le numéro avec regex
                    num_match = re.search(r'(?:\+228|228)?\s*(\d{2}[-\s]?\d{2}[-\s]?\d{2}[-\s]?\d{2})', text)
                    if num_match:
                        telephone = num_match.group(0).strip()
                    elif re.search(r'\d{8}', text):  # Numéro sans séparateur
                        telephone = text
            
            # Méthode 3: Chercher pattern WhatsApp
            if telephone == 'N/A':
                whatsapp = soup.find(['a', 'button'], href=re.compile(r'whatsapp|wa\.me'))
                if whatsapp:
                    wa_href = whatsapp.get('href', '')
                    # Extraire depuis wa.me/228XXXXXXXX
                    wa_match = re.search(r'(?:wa\.me/|whatsapp.*phone=)(\+?\d+)', wa_href)
                    if wa_match:
                        telephone = wa_match.group(1)
                    else:
                        telephone = whatsapp.get_text(strip=True)
            
            # Méthode 4: Chercher bouton "Appeler" ou "Contacter"
            if telephone == 'N/A':
                call_button = soup.find(['button', 'a'], string=re.compile(r'appeler|contacter|téléphone|whatsapp', re.I))
                if call_button:
                    # Chercher data-phone ou data-tel
                    tel_data = call_button.get('data-phone') or call_button.get('data-tel') or call_button.get('data-number')
                    if tel_data:
                        telephone = tel_data
                    else:
                        # Chercher dans onclick
                        onclick = call_button.get('onclick', '')
                        tel_match = re.search(r'(\+?228\d{8}|\d{8})', onclick)
                        if tel_match:
                            telephone = tel_match.group(1)
            
            # Méthode 5: Chercher dans tout le texte visible
            if telephone == 'N/A':
                # Patterns de numéros togolais
                phone_patterns = [
                    r'\+228\s*\d{2}[-\s]?\d{2}[-\s]?\d{2}[-\s]?\d{2}',  # +228 XX XX XX XX
                    r'228\s*\d{2}[-\s]?\d{2}[-\s]?\d{2}[-\s]?\d{2}',     # 228 XX XX XX XX
                    r'(?:^|\D)(\d{2}[-\s]?\d{2}[-\s]?\d{2}[-\s]?\d{2})(?:\D|$)',  # XX XX XX XX
                ]
                
                body_text = soup.get_text()
                for pattern in phone_patterns:
                    phone_match = re.search(pattern, body_text)
                    if phone_match:
                        if len(phone_match.groups()) > 0:
                            telephone = phone_match.group(1).strip()
                        else:
                            telephone = phone_match.group(0).strip()
                        break
            
            # Méthode 6: Chercher dans les scripts (parfois le numéro est en JS)
            if telephone == 'N/A':
                scripts = soup.find_all('script')
                for script in scripts:
                    script_text = script.string if script.string else ''
                    # Chercher pattern de numéro togolais dans le JS
                    js_match = re.search(r'(?:phone|tel|contact|whatsapp)["\']?\s*:\s*["\'](\+?228\d{8}|228\d{8}|\d{8})["\']', script_text, re.I)
                    if js_match:
                        telephone = js_match.group(1)
                        break
            
            # Nettoyer le numéro trouvé
            if telephone and telephone != 'N/A':
                # Supprimer les caractères non numériques sauf le +
                telephone = re.sub(r'[^\d+\s-]', '', telephone)
                # Normaliser les espaces
                telephone = re.sub(r'\s+', ' ', telephone).strip()
                # Si commence par 228 sans +, ajouter le +
                if telephone.startswith('228') and not telephone.startswith('+'):
                    telephone = '+' + telephone
                # Si c'est juste 8 chiffres, ajouter +228
                elif re.match(r'^\d{8}$', telephone.replace(' ', '').replace('-', '')):
                    telephone = '+228 ' + telephone
            
            details['contact'] = telephone if telephone != 'N/A' else 'Voir annonce'
            
            # Référence
            ref_match = re.search(r'/annonce/[^/]+/[^/]+-(\d+)', url)
            details['reference'] = ref_match.group(1) if ref_match else 'N/A'
            
            return details
            
        except Exception as e:
            return {'url': url, 'erreur': str(e)}
    
    def export_to_csv(self, filename=None):
        """Exporte les données vers CSV avec sauvegarde après chaque page"""
        if not self.all_listings:
            return
        
        if filename is None:
            filename = self.output_file
            
        print(f"\n💾 Sauvegarde dans {filename}...")
        
        df = pd.DataFrame(self.all_listings)
        
        columns_order = [
            'reference', 'titre', 'type_bien', 'type_offre', 'prix', 
            'nombre_pieces', 'surface', 
            'quartier', 'ville', 'region', 'localisation',
            'description', 'contact', 'date_publication', 'url'
        ]
        
        for col in columns_order:
            if col not in df.columns:
                df[col] = 'N/A'
        
        df = df[columns_order]
        
        # Renommer les colonnes en français
        df.columns = [
            'Référence', 'Titre', 'Type de Bien', 'Type Offre', 'Prix', 
            'Pièces', 'Surface', 
            'Quartier', 'Ville', 'Région', 'Localisation Complète',
            'Description', 'Contact', 'Date Publication', 'Lien Annonce'
        ]
        
        # Sauvegarder en CSV avec séparateur point-virgule
        df.to_csv(filename, index=False, encoding='utf-8-sig', sep=';')
        
        print(f"✅ Sauvegarde réussie - {len(df)} annonces")
    
    def scrape_first_5_pages(self):
        """Scrape avec sauvegarde après chaque page"""
        print("╔═══════════════════════════════════════════════════════════╗")
        print("║   SCRAPING 60 PAGES avec SAUVEGARDE INCRÉMENTALE         ║")
        print("╚═══════════════════════════════════════════════════════════╝\n")
        
        max_pages = 60
        total_listings = 0
        
        for page_num in range(1, max_pages + 1):
            current_url = f"{self.category_url}?page={page_num}"
            
            print(f"\n{'='*60}")
            print(f"📄 PAGE {page_num}/{max_pages}")
            print(f"{'='*60}")
            print(f"🔗 {current_url}")
            
            listings, _ = self.get_page_listings(current_url)
            
            if not listings:
                print(f"⚠️  Aucune annonce trouvée")
                continue
            
            print(f"✅ {len(listings)} annonces trouvées")
            print(f"⏳ Extraction en cours...", end=' ')
            
            extracted = 0
            for idx, listing in enumerate(listings, 1):
                if idx % 10 == 0:
                    print(f"{idx}", end='...')
                details = self.extract_listing_details(listing['url'])
                if details and 'erreur' not in details:
                    self.all_listings.append(details)
                    total_listings += 1
                    extracted += 1
            
            print(f"\n📊 Extraites: {extracted}")
            print(f"📊 Total: {total_listings}")
            
            # Sauvegarde après chaque page
            if self.all_listings:
                self.export_to_csv()
                print(f"💾 Données sauvegardées après page {page_num}")
            
            if page_num < max_pages:
                print(f"⏸️  Pause 2s...")
                time.sleep(2)
        
        print(f"\n{'='*60}")
        print(f"✅ TERMINÉ")
        print(f"{'='*60}")
        print(f"📊 Pages: {max_pages}")
        print(f"📊 Annonces: {total_listings}")
        print(f"📁 Fichier: {self.output_file}")
        print(f"{'='*60}\n")
        
        return self.all_listings

def main(debug=False):
    print("\n🚀 DÉMARRAGE...\n")
    
    import os
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        script_dir = os.getcwd()
    
    output_path = os.path.join(script_dir, 'coinafrique_60_pages.csv')
    
    scraper = CoinAfriqueScraperTogo(output_file=output_path, debug=debug)
    
    scraper.scrape_first_5_pages()
    
    if scraper.all_listings:
        print("🎉 TERMINÉ!")
        print(f"📁 Fichier: {output_path}")
    else:
        print("⚠️  Aucune donnée")

if __name__ == "__main__":
    main(debug=False)


🚀 DÉMARRAGE...

╔═══════════════════════════════════════════════════════════╗
║   SCRAPING 60 PAGES avec SAUVEGARDE INCRÉMENTALE         ║
╚═══════════════════════════════════════════════════════════╝


📄 PAGE 1/60
🔗 https://tg.coinafrique.com/categorie/immobilier?page=1
✅ 94 annonces trouvées
⏳ Extraction en cours... 10...20...30...40...50...60...70...80...90...
📊 Extraites: 90
📊 Total: 90

💾 Sauvegarde dans C:\Users\cyrid\Documents\Scrap_ahoe\coinafrique_60_pages.csv...
✅ Sauvegarde réussie - 90 annonces
💾 Données sauvegardées après page 1
⏸️  Pause 2s...

📄 PAGE 2/60
🔗 https://tg.coinafrique.com/categorie/immobilier?page=2
✅ 94 annonces trouvées
⏳ Extraction en cours... 10...20...30...40...50...60...70...80...90...
📊 Extraites: 94
📊 Total: 184

💾 Sauvegarde dans C:\Users\cyrid\Documents\Scrap_ahoe\coinafrique_60_pages.csv...
✅ Sauvegarde réussie - 184 annonces
💾 Données sauvegardées après page 2
⏸️  Pause 2s...

📄 PAGE 3/60
🔗 https://tg.coinafrique.com/categorie/immobilier?page=3
✅